## tool: KNNFeatureAggregator (5 баллов)

Нужно написать класс, который будет справляться с задачей генерации новых фичей по ближайшим соседям.
Принцип его работы объясним на примере. Допустим, мы находимся в каком-то пайплайне генерации признаков. Разберем псевдокод ниже:
```python
# 1
'''
    Создаем объект нашего класса - он принимает на вход информацию о том, какой будет индекс для поиска ближ. соседей.
    Далее, "обучаем" индекс, если это нужно делать (строим граф, строим ivf-табличку) и т.п.).
    После этого блока, у нас есть обученный индекс, готовый искать ближайших соседей по train_data.
'''
knn_feature_aggregator = KNNFeatureAggregator(index_info)
knn_feature_aggregator.train(train_data, index_add_info)

# 2
'''
    Считаем индексы ближайших соседей. На данном этапе мы хотим получить признаки для обучающей выборки, поэтому
        подаем в качестве query_data нашу обучалку.
    Указывам is_train=True, чтобы вернуть k ближайших соседей без учета самих себя (считая k+1 соседей + выкидывая 1 столбик).
    k указываем __МАКСИМАЛЬНОЕ_ИЗ_ТРЕБУЮЩИХСЯ_НИЖЕ__ (пока не анализируем что это значит, просто имеем в виду).

    Возвращает np.array размера (query_data.shape[0], k) с айдишниками ближ. соседей
'''
train_neighbors = knn_feature_aggregator.kneighbors(
        query_data=train_data,
        k=100,
        is_train=True,
        index_add_info=index_add_info
)

# 4 (сначала см. пункт 3 ниже)
'''
    Информацию о признаках можно подавать, например, в виде такого словаря.
    Ключи - названия результирующих колонок с новыми признаками.
    Значения - таплы из:
        1. Название оригинальной колонки, по которой агрегируемся
        2. Аггрегирующая фукнция
        3. Список из количества ближайших соседей, по которым считаем агг. функцию.
            Здесь каждое число должно быть НЕ БОЛЬШЕ k из пункта 2 (вспоминаем "__МАКСИМАЛЬНОЕ_ИЗ_ТРЕБУЮЩИХСЯ_НИЖЕ__", понимаем :)

    Пример:
        Имеем из п. 2 айдишники соседей:
        train_neighbors = array([[1, 2, 3],
                                 [2, 0, 3],
                                 [3, 1, 4],
                                 [4, 2, 1],
                                 [3, 2, 1]], dtype=uint64)

        Тогда по записи {
            ...
            'new_neighbors_age_mean': ('age', 'mean', [2, 3]),
        }

        Создадутся две новых колонки - 'new_neighbors_age_mean_2nn', 'new_neighbors_age_mean_3nn'.
        В первой будет для каждого объекта лежать средний возраст его двух ближ. соседей,
            во второй - средний возраст трех ближ. соседей.

'''
feature_info = {
    "new_col_name_1": ("original_col_name_1", lambda x: x.sum(), [10, 20, 100]),
    "new_col_name_2": ("original_col_name_1", lambda x: x.mean(), [11, 21, 101]),
    "new_col_name_3": ("original_col_name_2", lambda x: x.min() % 3, [50, 80, 100]),
}

# 3
'''
    Суть этого класса - генерировать новые фичи на основе ближайших соседей. Здесь мы это и делаем.
    Для этого подаем на вход айдишники соседей из обучающей выборки и саму обучающую выборку.
    Далее, подаем на вход информацию о том, "какие" признаки нам нужны, см. выше.

    Возвращает датафрейм размера (neighbor_ids.shape[0], количество_новых_фичей_по_feature_info)
'''
train_new_feature_df = knn_feature_aggregator.make_features(
    neighbor_ids=train_neighbors,
    train_data=train_data,
    feature_info=feature_info
)
train_data_with_new_features = merge(train_data, train_new_feature_df)

# 5
'''
    Для тестовой выборки пайплайн будет выглядеть аналогично, за исключением того, что is_train теперь False
'''
test_neighbors = knn_feature_aggregator.kneighbors(
        query_data=test_data,
        k=100,
        is_train=False,
        index_add_info=index_add_info
)
test_new_feature_df = knn_feature_aggregator.make_features(
    neighbor_ids=test_neighbors,
    train_data=train_data,
    feature_info=feature_info
)
test_data_with_new_features = merge(test_data, test_new_feature_df)

```

### Задание:
Написать класс, который реализует все, что описано выше, в частности:

**\_\_init\_\_**
- вы сами решаете, какой будет индекс, будет ли он фиксирован и т.п.

**train**
- обучающую выборку не нужно сохранять в объект класса в целях экономии памяти
- если вам нужно разбить `train` на `train` и `add_items`,
      чтобы поддерживать обучение индекса на репрезентативном сабсэмпле, можете это сделать
- аргумент train_data - не обязательно выборка со всеми признаками.
      Вы хотите подавать сюда то подмножество признаков, по которому будете искать соседей
      (соответственно, нужно подавать уже приведенные к однородному виду данные)

**kneighbors**
- обязательна поддержка флажка is_train с описанным выше функционалом
- аргумент query_data - см. замечание к аргументу train_data из метода train выше

**make_features**
- обработайте отдельно случай, когда вы в качестве ближайших соседей подаете единственное число.
      Не нужно извне подавать список из одного числа, обработка должна быть внутри

**Эффективность**

Все должно быть реализовано эффективно. В том числе:
- без цикла for по всем объектам train_data/query_data
- без pd.DataFrame.apply
- можно использовать np.apply_along_axis (работает в ~5 раз быстрее, чем pandas)

**Пример**

Нужно привести пример работы вашего класса, запустив ячейки в блоке "Пример" ниже.
Не удаляйте авторский пример!

**Вопросы**

Нужно ответить на вопросы в блоке "Вопросы" ниже

**Note:** feature_info можете реализовать в любом виде, но описанный выше способ хорош тем,
      что его легко привести в удобный для дальнейшей работы вид:

In [1]:
from typing import Dict, Tuple, List
import numpy as np
import pandas as pd
import pynndescent

feature_info = {
    "new_col_name_1": ("original_col_name_1", lambda x: x.sum(), [10, 20, 100]),
    "new_col_name_2": ("original_col_name_1", lambda x: x.mean(), [11, 21, 101]),
    "new_col_name_3": ("original_col_name_2", lambda x: x.min() % 3, [50, 80, 100]),
}
pd.DataFrame(feature_info, index=["col_name", "func", "k"]).T.explode("k").reset_index(
    names="new_col"
)

,new_col,col_name,func,k
0,new_col_name_1,original_col_name_1,<function <lambda> at 0x7fe57ef6cc20>,10
1,new_col_name_1,original_col_name_1,<function <lambda> at 0x7fe57ef6cc20>,20
2,new_col_name_1,original_col_name_1,<function <lambda> at 0x7fe57ef6cc20>,100
3,new_col_name_2,original_col_name_1,<function <lambda> at 0x7fe55bfc3d80>,11
4,new_col_name_2,original_col_name_1,<function <lambda> at 0x7fe55bfc3d80>,21
5,new_col_name_2,original_col_name_1,<function <lambda> at 0x7fe55bfc3d80>,101
6,new_col_name_3,original_col_name_2,<function <lambda> at 0x7fe55c57aa20>,50
7,new_col_name_3,original_col_name_2,<function <lambda> at 0x7fe55c57aa20>,80
8,new_col_name_3,original_col_name_2,<function <lambda> at 0x7fe55c57aa20>,100


In [2]:
# для быстрого поиска ближайших соседей
# будем использовать pynndescent
class KNNFeatureAggregator:
    def __init__(
        self,
        metric: str = "euclidean",
        n_neighbors: int = 3,
        epsilon: float = 0.1,
        pruning_degree_multiplier: float = 1.5,
        diversify_prob: float = 1.0,
    ):
        # гиперпараметры алгоритма в инициализации
        self.metric = metric
        self.n_neighbors = n_neighbors
        self.epsilon = epsilon
        self.pruning_degree_multiplier = pruning_degree_multiplier
        self.diversify_prob = diversify_prob
        self.index = None

    def train(self, data: pd.DataFrame):
        # обучение индекса в .train()
        self.index = pynndescent.NNDescent(
            data,
            metric=self.metric,
            n_neighbors=self.n_neighbors,
            pruning_degree_multiplier=self.pruning_degree_multiplier,
            diversify_prob=self.diversify_prob,
        )
        self.index.prepare()

    def kneighbors(self, query: pd.DataFrame, k: int, is_train: bool = False):
        # поиск ближайших соседей в .kneighbors()
        # если is_train=True, то возвращаем k, за исключением самого первого
        if is_train:
            ids, _ = self.index.query(query, k=k + 1, epsilon=0.0)
            ids = ids[:, 1:]
        else:
            ids, _ = self.index.query(query, k=k, epsilon=self.epsilon)
        return ids

    def make_features(
        self,
        data: pd.DataFrame,
        neighbor_ids: np.ndarray,
        feature_info: Dict[str, Tuple[str, str, List[int]]],
    ) -> pd.DataFrame:
        # каждая новая фича описывается строкой в этом датафрейме:
        feature_df = (
            pd.DataFrame(feature_info, index=["col_name", "func", "k"])
            .T.explode("k")
            .reset_index(names="new_col_name")
        )
        new_df = pd.DataFrame()
        for _, row in feature_df.iterrows():
            new_col_name, col_name, func, k = (
                row["new_col_name"],
                row["col_name"],
                row["func"],
                row["k"],
            )
            # записываем матрицу (data.shape[0], k) в neighbor_data
            # в каждой строке - значения фичи col_name для k ближайших соседей
            neighbor_data = (
                data[col_name]
                .iloc[neighbor_ids[:, :k].flatten()]
                .to_numpy()
                .reshape(-1, k)
            )

            # чтобы посчитать func по соседям, для каждой строки
            # из значений соседей применяем эту функцию 
            new_df[f"{new_col_name}@{k}"] = np.apply_along_axis(
                func, axis=1, arr=neighbor_data
            )

        return new_df

### Пример

Ваш:

In [3]:
train_data = pd.DataFrame({
    'a': [1, 2, 3, 4, 5],
    'b': [10, 19, 27, 34, 40]
})
agg = KNNFeatureAggregator()
agg.train(train_data)
neighbor_ids = agg.kneighbors(train_data, is_train=True, k=3)
neighbor_ids # у вас индексы ближ. соседей могут отличаться

array([[1, 2, 3],
       [2, 0, 3],
       [3, 1, 4],
       [4, 2, 1],
       [3, 2, 1]], dtype=int32)

In [4]:
X = agg.make_features(
    train_data,
    neighbor_ids,
    feature_info={
        "a_sum": ("a", lambda x: x.sum(), [2, 3]),
        "b_whatever": ("b", lambda x: x.min(), 2),
    },
)
X

,a_sum@2,a_sum@3,b_whatever@2
0,5,9,19
1,4,8,10
2,6,11,19
3,8,10,27
4,7,9,27


Авторский:

In [20]:
train_data = pd.DataFrame({
    'a': [1, 2, 3, 4, 5],
    'b': [10, 19, 27, 34, 40]
})
agg = KNNFeatureAgg(dim=2, metric='l2') # у автора: hnsw index
agg.train(train_data)
neighbor_ids = agg.kneighbors(train_data, is_train=True, k=3)
neighbor_ids # у вас индексы ближ. соседей могут отличаться

array([[1, 2, 3],
       [2, 0, 3],
       [3, 1, 4],
       [4, 2, 1],
       [3, 2, 1]], dtype=uint64)

In [21]:
X = agg.make_features(neighbor_ids, feature_info={
    'a_sum': ('a', lambda x: x.sum(), [2, 3]),
    'b_whatever': ('b', lambda x: x.min(), 2),
})
X

,a_sum_2nn,b_whatever_2nn,a_sum_3nn
0,5,19,9
1,4,10,8
2,6,19,11
3,8,27,10
4,7,27,9


### Вопросы

1) Какой / какие индекс[-ы] вы решили использовать для этой задачи и почему?
2) Какие недостатки / потенциальные зоны для улучшения у вашей текущей реализации?

### Ответы

1) Использовался `PyNNDescent`, так как он предоставляет возможность контролировать точность и скорость поиска ближайших соседей своими параметрами с удобным интерфейсом.
2) Текущая реализация адаптирована только под `pandas.DataFrame` структуру данных, так что возможны улучшения этого решения, если реализовать подобный функционал например, на разреженные данные